# Gold model

In [14]:
# ── Imports ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import timezone, datetime
from sqlalchemy import create_engine, text
import urllib
import os
from dotenv import load_dotenv

In [5]:
# ── Paths ─────────────────────────────────────────────────────────────
ROOT_DIR     = Path("..").resolve()
SILVER_DIR   = ROOT_DIR / "data" / "silver"
GOLD_DIR     = ROOT_DIR / "data" / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)

WC_FILE      = SILVER_DIR / "silver_wc_output.csv"
SOCIO_FILE   = SILVER_DIR / "valid_socioeconomics.csv"

print(f"WC file   : {WC_FILE.name}")
print(f"Socio file: {SOCIO_FILE.name}")

WC file   : silver_wc_output.csv
Socio file: valid_socioeconomics.csv


In [16]:
# ── SQL Server connection ──────────────────────────────────────────────
# Credentials loaded from .env — never hardcoded
load_dotenv()

SERVER   = "sebastien-sql.database.windows.net"
DATABASE = os.getenv("SQL_DATABASE")  
USERNAME = "sqladmin"
PASSWORD = os.getenv("SQL_PASSWORD")  

conn_str = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    f"UID={USERNAME};"
    f"PWD={PASSWORD};"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)
params = urllib.parse.quote_plus(conn_str)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# Test connection
with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
print("Connection OK")

Connection OK


In [17]:
# ── Load silver data ───────────────────────────────────────────────────
df_wc    = pd.read_csv(WC_FILE)
df_socio = pd.read_csv(SOCIO_FILE)

print("WC shape    :", df_wc.shape)
print("Socio shape :", df_socio.shape)
print("\nWC columns :", df_wc.columns.tolist())
print("Socio columns:", df_socio.columns.tolist())

WC shape    : (384, 14)
Socio shape : (5158, 7)

WC columns : ['year', 'country', 'city', 'stage', 'home_team', 'away_team', 'home_score', 'away_score', 'winning_team', 'losing_team', 'date_clean', 'source_name', 'load_timestamp_clean', 'run_id']
Socio columns: ['country_name', 'country_code', 'year', 'population', 'gdp_per_capita_usd', 'source_name', 'load_timestamp']


In [18]:
# ── Unpivot matches: wide → long ──────────────────────────────────────
# Chaque match génère 2 lignes : 1 pour home team, 1 pour away team

home = df_wc.copy()
home["team_country"]   = home["home_team"]
home["goals_scored"]   = home["home_score"]
home["goals_conceded"] = home["away_score"]

away = df_wc.copy()
away["team_country"]   = away["away_team"]
away["goals_scored"]   = away["away_score"]
away["goals_conceded"] = away["home_score"]

df_long = pd.concat([home, away], ignore_index=True)

# goal_difference
df_long["goal_difference"] = df_long["goals_scored"] - df_long["goals_conceded"]

# outcome
df_long["outcome"] = "draw"
df_long.loc[df_long["team_country"] == df_long["winning_team"], "outcome"] = "win"
df_long.loc[df_long["team_country"] == df_long["losing_team"],  "outcome"] = "loss"

print(f"Rows after unpivot : {df_long.shape}")
print(df_long[["year", "team_country", "goals_scored", "goals_conceded", "outcome"]].head(6))

Rows after unpivot : (768, 19)
   year team_country  goals_scored  goals_conceded outcome
0  1998       Brazil             2               1     win
1  1998      Morocco             2               2    draw
2  1998     Cameroon             1               1    draw
3  1998        Italy             2               2    draw
4  1998       France             3               0     win
5  1998     Paraguay             0               0    draw


In [19]:
# ── is_host flag ───────────────────────────────────────────────────────
# Compare team_country with country (host_country) to create a binary flag indicating if the team is the host of the match
df_long["is_host"] = (df_long["team_country"] == df_long["country"]).astype(int)

print(f"Matches with is_host=1 : {df_long['is_host'].sum()}")
print(df_long[df_long["is_host"] == 1][["year", "team_country", "country"]].drop_duplicates())

Matches with is_host=1 : 40
     year  team_country       country
4    1998        France        France
76   2002         Japan         Japan
77   2002   South Korea   South Korea
128  2006       Germany       Germany
192  2010  South Africa  South Africa
256  2014        Brazil        Brazil
320  2018        Russia        Russia


In [20]:
print(df_long["stage"].unique())

<StringArray>
[      'Group A',       'Group B',       'Group C',       'Group D',
       'Group E',       'Group H',       'Group F',       'Group G',
   'Round of 16', 'Quarterfinals',    'Semifinals',   'Third place',
         'Final',      'Group D ']
Length: 14, dtype: str


In [21]:
# ── Performance stars ─────────────────────────────────────────────────
# Strip whitespace from stage to handle 'Group D ' trailing space
df_long["stage"] = df_long["stage"].str.strip()

STAGE_STARS = {
    "Group A": 1, "Group B": 1, "Group C": 1, "Group D": 1,
    "Group E": 1, "Group F": 1, "Group G": 1, "Group H": 1,
    "Round of 16": 2,
    "Quarterfinals": 3,
    "Semifinals": 4,
    "Third place": 4,  # semi-final losers
    "Final": 5,        # overridden to 6 for winner below
}

df_long["stage_star"] = df_long["stage"].map(STAGE_STARS).fillna(1)

# Final winner → 6 stars
df_long.loc[
    (df_long["stage"] == "Final") & (df_long["outcome"] == "win"),
    "stage_star"
] = 6

# performance_stars = max stage reached per team × year
perf = (
    df_long.groupby(["team_country", "year"])["stage_star"]
    .max()
    .reset_index()
    .rename(columns={"stage_star": "performance_stars"})
)

df_long = df_long.merge(perf, on=["team_country", "year"], how="left")
df_long = df_long.drop(columns=["stage_star"])

print("Performance stars distribution:")
print(df_long[["team_country", "year", "performance_stars"]].drop_duplicates()["performance_stars"].value_counts().sort_index())

Performance stars distribution:
performance_stars
1    96
2    48
3    24
4    12
5     6
6     6
Name: count, dtype: int64


In [22]:
# ── Join socioeconomics (population + gdp_per_capita_usd) ─────────────
# Join key: team_country + year
df_long = df_long.merge(
    df_socio[["country_name", "year", "population", "gdp_per_capita_usd"]],
    left_on=["team_country", "year"],
    right_on=["country_name", "year"],
    how="left"
).drop(columns=["country_name"])

missing_socio = df_long[df_long["population"].isna()]["team_country"].unique()
print(f"Teams missing socioeconomics: {len(missing_socio)}")
print(missing_socio)

Teams missing socioeconomics: 1
<StringArray>
['North Korea']
Length: 1, dtype: str


In [23]:
# ── Assemble fact_wc_match ────────────────────────────────────────────
fact = df_long[[
    "year", "country", "team_country",
    "goals_scored", "goals_conceded", "goal_difference",
    "outcome", "stage", "performance_stars",
    "is_host", "population", "gdp_per_capita_usd"
]].copy()

fact = fact.rename(columns={
    "country":      "host_country",
    "team_country": "team_country"
})

print(f"fact_wc_match shape: {fact.shape}")
fact.head()

fact_wc_match shape: (768, 12)


,year,host_country,team_country,goals_scored,goals_conceded,goal_difference,outcome,stage,performance_stars,is_host,population,gdp_per_capita_usd
0,1998,France,Brazil,2,1,1,win,Group A,5,0,169159655.0,8.637110e+11
1,1998,France,Morocco,2,2,0,draw,Group A,1,0,27621648.0,4.649761e+10
2,1998,France,Cameroon,1,1,0,draw,Group B,1,0,14144860.0,1.129814e+10
3,1998,France,Italy,2,2,0,draw,Group B,3,0,56906744.0,1.272730e+12
4,1998,France,France,3,0,3,win,Group C,6,1,60190684.0,1.496906e+12


In [ ]:
# ── FK mappings ───────────────────────────────────────────────────────
# These must match exactly what the dim table owners generate
# Share this logic with your team

year_map = {y: i+1 for i, y in enumerate(sorted(fact["year"].unique()))}
# {1998: 1, 2002: 2, 2006: 3, 2010: 4, 2014: 5, 2018: 6}

tournament_map = {y: i+1 for i, y in enumerate(sorted(fact["year"].unique()))}
# Same grain as dim_tournament

all_countries = sorted(set(fact["host_country"].unique()) | set(fact["team_country"].unique()))
country_map = {c: i+1 for i, c in enumerate(all_countries)}

fact["year_id"] = fact["year"].map(year_map)
fact["tournament_id"] = fact["year"].map(tournament_map)
fact["host_country_id"] = fact["host_country"].map(country_map)
fact["team_country_id"] = fact["team_country"].map(country_map)

print("year_map :", year_map)
print("\nSample country_map (first 5):")
print(dict(list(country_map.items())[:5]))

year_map : {np.int64(1998): 1, np.int64(2002): 2, np.int64(2006): 3, np.int64(2010): 4, np.int64(2014): 5, np.int64(2018): 6}

Sample country_map (first 5):
{'Algeria': 1, 'Angola': 2, 'Argentina': 3, 'Australia': 4, 'Austria': 5}


In [25]:
# ── PK + final column selection ───────────────────────────────────────
fact = fact.reset_index(drop=True)
fact.insert(0, "match_country_id", range(1, len(fact) + 1))

fact_final = fact[[
    "match_country_id",
    "year_id", "tournament_id", "host_country_id", "team_country_id",
    "goals_scored", "goals_conceded", "goal_difference",
    "outcome", "stage", "performance_stars",
    "is_host", "population", "gdp_per_capita_usd"
]]

print(f"Final shape : {fact_final.shape}")
print(f"Null check  :\n{fact_final.isnull().sum()}")
fact_final.head()

Final shape : (768, 14)
Null check  :
match_country_id      0
year_id               0
tournament_id         0
host_country_id       0
team_country_id       0
goals_scored          0
goals_conceded        0
goal_difference       0
outcome               0
stage                 0
performance_stars     0
is_host               0
population            3
gdp_per_capita_usd    3
dtype: int64


,match_country_id,year_id,tournament_id,host_country_id,team_country_id,goals_scored,goals_conceded,goal_difference,outcome,stage,performance_stars,is_host,population,gdp_per_capita_usd
0,1,1,1,21,8,2,1,1,win,Group A,5,0,169159655.0,8.637110e+11
1,2,1,1,21,33,2,2,0,draw,Group A,1,0,27621648.0,4.649761e+10
2,3,1,1,21,10,1,1,0,draw,Group B,1,0,14144860.0,1.129814e+10
3,4,1,1,21,28,2,2,0,draw,Group B,3,0,56906744.0,1.272730e+12
4,5,1,1,21,21,3,0,3,win,Group C,6,1,60190684.0,1.496906e+12


In [26]:
# ── Load to SQL Server ────────────────────────────────────────────────
TABLE_NAME = "fact_wc_match"

fact_final.to_sql(
    TABLE_NAME,
    engine,
    if_exists="replace",   # DROP + INSERT
    index=False,
    schema="dbo"
)

# Verify
with engine.connect() as conn:
    count = conn.execute(text(f"SELECT COUNT(*) FROM dbo.{TABLE_NAME}")).scalar_one()

print(f"Loaded {count} rows into dbo.{TABLE_NAME}")

Loaded 768 rows into dbo.fact_wc_match
